# Ring Attention / 序列并行：四组可复现实验

对应文章：[分布式训练（05）序列并行——把注意力沿序列切开，Ring Attention 与长上下文](https://lrypcy.github.io/2026/08/31/dist-train-05-sequence-parallel/)

全部纯 numpy、纯 CPU、固定 seed。四组实验各自独立，每组末尾带断言，最后一个 cell 把断言总数打出来。

| 组 | 回答什么 |
| --- | --- |
| A | 分块 online softmax 与全局 softmax **是否逐元素等价**；「每块各自归一再拼」错多少 |
| B | 因果掩码下三种切法（contiguous / zigzag / striped）的**墙钟**与利用率 |
| C | Ring / all-gather / Ulysses 三种通信方案的**每卡字节数** |
| D | 通信藏进计算所需的**最小块长与最小序列长度**，与论文 Table 2 对账 |

In [1]:
import math

import numpy as np

from ring_attention_lib import (
    HARDWARE,
    allgather_comm_bytes,
    global_attention,
    min_block_len,
    min_seq_len,
    naive_blockwise_attention,
    ring_attention,
    ring_comm_bytes,
    simulate_wallclock,
    token_owner,
    ulysses_comm_bytes,
)

SEED = 0
rng = np.random.default_rng(SEED)
N_ASSERT = 0


def check(cond, msg):
    global N_ASSERT
    assert cond, msg
    N_ASSERT += 1
    print(f"[OK] {msg}")


print("numpy", np.__version__, "| seed", SEED)

numpy 2.1.1 | seed 0


## A. 分块 online softmax 与全局 softmax 逐元素等价

Ring Attention 在环上每轮只做一件事：把新到的 KV 块的部分 softmax 状态 $$(m, l, O)$$ 与本地累加器合并。
合并不是「加权平均」，而是把各自的 max 拉到同一个尺度后相加：

$$
m=\max(m_a,m_b),\quad l=l_a e^{m_a-m}+l_b e^{m_b-m},\quad O=O_a e^{m_a-m}+O_b e^{m_b-m}
$$

如果这个式子成立，那么任意分块顺序、任意块大小，最终 $$O/l$$ 都与一次性全局 softmax 相同 —— 这是 ring
允许「环序任意」的全部依据，也是它与近似方法的分界线。

In [2]:
S, d, NB = 512, 64, 8
Q = rng.normal(size=(S, d))
K = rng.normal(size=(S, d))
V = rng.normal(size=(S, d))

ref = global_attention(Q, K, V)
O_ring, trace = ring_attention(Q, K, V, NB)

err_ring = np.abs(O_ring - ref).max()
print(f"块数 NB={NB}，每块 {S // NB} token，ring vs 全局 softmax 最大绝对误差 {err_ring:.3e}")
check(err_ring < 1e-12, "分块 online 合并与全局 softmax 逐元素等价（float64 误差 < 1e-12）")

# 环序任意：五个随机置换，结果必须一致
errs = []
for i in range(5):
    perm = [int(x) for x in np.random.default_rng(100 + i).permutation(NB)]
    errs.append(np.abs(ring_attention(Q, K, V, NB, order=perm)[0] - ref).max())
print("五种环序的最大误差：", ", ".join(f"{e:.2e}" for e in errs))
check(max(errs) < 1e-12, "KV 块在环上的到达次序不影响结果（合并满足交换律）")

块数 NB=8，每块 64 token，ring vs 全局 softmax 最大绝对误差 4.996e-16
[OK] 分块 online 合并与全局 softmax 逐元素等价（float64 误差 < 1e-12）
五种环序的最大误差： 4.44e-16, 5.55e-16, 4.44e-16, 5.55e-16, 4.44e-16
[OK] KV 块在环上的到达次序不影响结果（合并满足交换律）


In [3]:
# 反例：每块各自 softmax、再按块数平均 —— 分块最容易写错的一种
err_naive = np.abs(naive_blockwise_attention(Q, K, V, NB) - ref).max()
print(f"「逐块归一后平均」的最大误差 {err_naive:.4f}")
print(f"同一批数据上正确实现的误差 {err_ring:.2e}，相差约 {err_naive / err_ring:.1e} 倍")
check(err_naive > 0.1, "不做全局归一的分块拼接是错的，误差 O(1) 而非 O(1e-16)")

# 数值稳定性：把 logits 整体放大 20 倍，朴素 exp 会溢出，online 合并不会
Q_big = Q * 20.0
ref_big = global_attention(Q_big, K, V)
O_big = ring_attention(Q_big, K, V, NB)[0]
print(f"logits 放大 20 倍后：全局 max = {(Q_big @ K.T / math.sqrt(d)).max():.1f}，"
      f"ring 误差 {np.abs(O_big - ref_big).max():.3e}")
check(np.abs(O_big - ref_big).max() < 1e-10, "每步减去 running max 后，极端 logits 下仍然稳定")

「逐块归一后平均」的最大误差 0.1864
同一批数据上正确实现的误差 5.00e-16，相差约 3.7e+14 倍
[OK] 不做全局归一的分块拼接是错的，误差 O(1) 而非 O(1e-16)
logits 放大 20 倍后：全局 max = 98.8，ring 误差 3.109e-15
[OK] 每步减去 running max 后，极端 logits 下仍然稳定


## B. 因果掩码下的负载分布：为什么需要 zigzag / striped

因果掩码让「序列前半段的 query 只看很少的 key，后半段看很多」。ring 每轮同步一次，**墙钟由最慢的那台决定**，
所以只要有一台在算整块，全环就得等 —— 而 contiguous 切法下每一轮都恰好有一台在算整块。

三种切法：

* **contiguous**：设备 $$r$$ 拿连续的第 $$r$$ 段（Ring Attention 原始做法）
* **zigzag**：序列切成 $$2P$$ 块，设备 $$r$$ 拿第 $$r$$ 块和倒数第 $$r$$ 块（Megatron 做法）
* **striped**：设备 $$r$$ 拿所有下标 $$\equiv r \pmod P$$ 的 token（Striped Attention 做法）

这里按 **token 对**精确计数：query $$q$$ 能看 key $$k$$ 当且仅当 $$q\ge k$$，工作量就是有效对的数量。

In [4]:
rows = []
for S_len, P in [(256, 4), (1024, 8), (4096, 16)]:
    base = None
    for scheme in ["contiguous", "zigzag", "striped"]:
        wall, per_dev, ideal = simulate_wallclock(S_len, P, scheme)
        if base is None:
            base = wall
        rows.append(
            dict(S=S_len, P=P, scheme=scheme, wall=wall,
                 ideal=ideal, util=ideal / wall, speedup=base / wall)
        )

print(f"{'序列':>6} {'卡数':>4} {'切法':>12} {'墙钟':>9} {'理想':>9} {'利用率':>7} {' vs 连续':>8}")
for r in rows:
    print(f"{r['S']:>6} {r['P']:>4} {r['scheme']:>12} {r['wall']:>9} {r['ideal']:>9.0f} "
          f"{r['util']:>7.1%} {r['speedup']:>7.2f}x")

    序列   卡数           切法        墙钟        理想     利用率    vs 连续
   256    4   contiguous     14368      8224   57.2%    1.00x
   256    4       zigzag      8224      8224  100.0%    1.75x
   256    4      striped      8320      8224   98.8%    1.73x
  1024    8   contiguous    122944     65600   53.4%    1.00x
  1024    8       zigzag     65600     65600  100.0%    1.87x
  1024    8      striped     66048     65600   99.3%    1.86x
  4096   16   contiguous   1015936    524416   51.6%    1.00x
  4096   16       zigzag    524416    524416  100.0%    1.94x
  4096   16      striped    526336    524416   99.6%    1.93x


In [5]:
def pick(S_len, P, scheme, key):
    for r in rows:
        if r["S"] == S_len and r["P"] == P and r["scheme"] == scheme:
            return r[key]
    raise KeyError((S_len, P, scheme))


# 连续切法：卡数越多越接近 50% 利用率（理论上限 2x）
util_c4 = pick(256, 4, "contiguous", "util")
util_c16 = pick(4096, 16, "contiguous", "util")
print(f"连续切法利用率：P=4 → {util_c4:.1%}，P=16 → {util_c16:.1%}（趋近 50%）")
check(util_c16 < util_c4 < 0.65, "连续切法利用率低于 2/3，且卡数越多越差")

# zigzag：2P 块时每卡的两块工作量互补，完全均衡
zig_utils = [r["util"] for r in rows if r["scheme"] == "zigzag"]
print("zigzag 各配置利用率：", ", ".join(f"{u:.4f}" for u in zig_utils))
check(min(zig_utils) > 0.999, "zigzag 把每卡工作量配成严格相等（利用率 = 100%）")

# striped：块内仍有一半被掩码，但每轮每卡都在算，接近均衡
str_utils = [r["util"] for r in rows if r["scheme"] == "striped"]
print("striped 各配置利用率：", ", ".join(f"{u:.4f}" for u in str_utils))
check(min(str_utils) > 0.95, "striped 的残余不均小于 5%")

sp16 = pick(4096, 16, "zigzag", "speedup")
sp16s = pick(4096, 16, "striped", "speedup")
print(f"P=16 时相对连续切法的墙钟加速：zigzag {sp16:.2f}x，striped {sp16s:.2f}x")
check(1.8 < sp16 < 2.0, "zigzag 的加速比逼近理论上限 2x（Striped Attention 论文给出的极限）")

连续切法利用率：P=4 → 57.2%，P=16 → 51.6%（趋近 50%）
[OK] 连续切法利用率低于 2/3，且卡数越多越差
zigzag 各配置利用率： 1.0000, 1.0000, 1.0000
[OK] zigzag 把每卡工作量配成严格相等（利用率 = 100%）
striped 各配置利用率： 0.9885, 0.9932, 0.9964
[OK] striped 的残余不均小于 5%
P=16 时相对连续切法的墙钟加速：zigzag 1.94x，striped 1.93x
[OK] zigzag 的加速比逼近理论上限 2x（Striped Attention 论文给出的极限）


In [6]:
# 用具体的分配表看清 zigzag 做了什么：P=4、16 token
owner_c = token_owner(16, 4, "contiguous")
owner_z = token_owner(16, 4, "zigzag")
print("contiguous 的 token 归属：", owner_c.tolist())
print("zigzag      的 token 归属：", owner_z.tolist())
check(owner_z.tolist() == [0, 0, 1, 1, 2, 2, 3, 3, 3, 3, 2, 2, 1, 1, 0, 0],
      "zigzag 的归属是 [0,1,2,3,3,2,1,0] 的镜像串接：前段越靠后负载越重，后段越靠前负载越轻")

contiguous 的 token 归属： [0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3]
zigzag      的 token 归属： [0, 0, 1, 1, 2, 2, 3, 3, 3, 3, 2, 2, 1, 1, 0, 0]
[OK] zigzag 的归属是 [0,1,2,3,3,2,1,0] 的镜像串接：前段越靠后负载越重，后段越靠前负载越轻


## C. 通信量：ring 与 all-gather 同量级，Ulysses 少一个 $$P$$ 倍

设序列长 $$S$$、KV 全宽 $$h$$（所有 head 拼起来）、并行度 $$P$$、bf16：

* **ring**：每卡发送 $$P-1$$ 个 KV 块，每块 $$2\cdot\frac{S}{P}\cdot h$$ 个元素
* **all-gather**：先把全部 KV 收齐，每卡收到全量的 $$\frac{P-1}{P}$$
* **Ulysses**：每层两次 all-to-all，聚合量 $$3Sh$$（QKV）+ $$Sh$$（输出），每链路摊 $$\frac{1}{P}$$

前两者**完全相等**——这是最容易被写错的一点：ring 省的从来不是带宽，而是**可重叠性**。

In [7]:
S_c, h, dtype_bytes = 131072, 8192, 2  # 128K 上下文、70B 级宽度
print(f"{'P':>3} {'ring (GB)':>11} {'all-gather (GB)':>16} {'Ulysses (GB)':>13} {'ring/Ulysses':>13}")
comm_rows = []
for P in [2, 4, 8, 16, 32]:
    r_ = ring_comm_bytes(S_c, h, P, dtype_bytes) / 1e9
    a_ = allgather_comm_bytes(S_c, h, P, dtype_bytes) / 1e9
    u_ = ulysses_comm_bytes(S_c, h, P, dtype_bytes) / 1e9
    comm_rows.append((P, r_, a_, u_))
    print(f"{P:>3} {r_:>11.1f} {a_:>16.1f} {u_:>13.1f} {r_ / u_:>12.1f}x")

  P   ring (GB)  all-gather (GB)  Ulysses (GB)  ring/Ulysses
  2         2.1              2.1           4.3          0.5x
  4         3.2              3.2           2.1          1.5x
  8         3.8              3.8           1.1          3.5x
 16         4.0              4.0           0.5          7.5x
 32         4.2              4.2           0.3         15.5x


In [8]:
ratios = [abs(r_ / a_ - 1) for _, r_, a_, _ in comm_rows]
print("ring 与 all-gather 的相对差：", ", ".join(f"{x:.1e}" for x in ratios))
check(max(ratios) < 1e-12, "ring 与 all-gather 每卡字节数完全相同（差异只在能否与计算重叠）")

for P, r_, a_, u_ in comm_rows:
    expect = (P - 1) / 2 if P > 1 else float("inf")
    got = r_ / u_ if u_ > 0 else float("inf")
    check(abs(got - expect) / expect < 1e-12,
          f"P={P} 时 ring/Ulysses = (P-1)/2 = {expect:.1f}（Ulysses 随 P 线性变便宜）")

ring 与 all-gather 的相对差： 0.0e+00, 0.0e+00, 0.0e+00, 0.0e+00, 0.0e+00
[OK] ring 与 all-gather 每卡字节数完全相同（差异只在能否与计算重叠）
[OK] P=2 时 ring/Ulysses = (P-1)/2 = 0.5（Ulysses 随 P 线性变便宜）
[OK] P=4 时 ring/Ulysses = (P-1)/2 = 1.5（Ulysses 随 P 线性变便宜）
[OK] P=8 时 ring/Ulysses = (P-1)/2 = 3.5（Ulysses 随 P 线性变便宜）
[OK] P=16 时 ring/Ulysses = (P-1)/2 = 7.5（Ulysses 随 P 线性变便宜）
[OK] P=32 时 ring/Ulysses = (P-1)/2 = 15.5（Ulysses 随 P 线性变便宜）


In [9]:
# 换成按层的口径：ring 每层每卡传 (P-1)/P * 2Sh，与 P 无关地趋近 2Sh
S_l, h_l, P_l = 131072, 8192, 8
per_layer_gb = ring_comm_bytes(S_l, h_l, P_l) / 1e9
n_layers = 80
print(f"128K 上下文、h={h_l}、P={P_l}：ring 每层每卡 {per_layer_gb:.1f} GB，"
      f"{n_layers} 层合计 {per_layer_gb * n_layers:.0f} GB/step")
print(f"同配置 all-gather 每层每卡 {allgather_comm_bytes(S_l, h_l, P_l) / 1e9:.1f} GB")
check(per_layer_gb * n_layers > 200, "128K 下 ring 每步的通信量是数百 GB：单机带宽是硬约束，不是优化项")

128K 上下文、h=8192、P=8：ring 每层每卡 3.8 GB，80 层合计 301 GB/step
同配置 all-gather 每层每卡 3.8 GB
[OK] 128K 下 ring 每步的通信量是数百 GB：单机带宽是硬约束，不是优化项


## D. 通信藏进计算：最小块长 $$c\ge F/B$$

ring 想「零开销」，必须让一块的注意力计算时间不短于这一块 KV 的传输时间。设块长 $$c$$、head 维 $$d$$、
单卡算力 $$F$$、单向带宽 $$B$$：

* 计算：$$4dc^2$$ FLOPs（$$QK^\top$$ 与 $$PV$$ 各 $$2dc^2$$）
* 通信：$$4cd$$ 字节（一对 KV 块，bf16）

令计算 $$\ge$$ 通信得 $$c\ge F/B$$。再加上「每卡要同时驻留 6 个块」，得到 $$s\ge 6c$$。

这就是 Ring Attention 论文 Table 2 的来历，也是它「单机内几乎免费、跨机就贵得离谱」的定量原因。

In [10]:
print(f"{'硬件':>18} {'F (TF)':>8} {'B (GB/s)':>9} {'c_min':>8} {'s_min':>9}")
hw_rows = []
for name, (F, B) in HARDWARE.items():
    c = min_block_len(F * 1e12, B * 1e9)
    s = min_seq_len(F * 1e12, B * 1e9)
    hw_rows.append((name, F, B, c, s))
    print(f"{name:>18} {F:>8.0f} {B:>9.1f} {c:>8.0f} {s:>9.0f}")

                硬件   F (TF)  B (GB/s)    c_min     s_min
     A100 + NVLink      312     300.0     1040      6240
 A100 + InfiniBand      312      12.5    24960    149760
            TPU v4      275     268.0     1026      6157
     H100 + NVLink      989     450.0     2198     13187
    H100 + IB 400G      989      25.0    39560    237360
    B200 + NVLink5     2250     900.0     2500     15000


In [11]:
d_ = dict((n, (F, B, c, s)) for n, F, B, c, s in hw_rows)

# 与 Ring Attention 论文 Table 2 对账（论文给 A100 NVLink 1.0K/6.2K、A100 IB 24.5K/149.5K）
_, _, c_nv, s_nv = d_["A100 + NVLink"]
_, _, c_ib, s_ib = d_["A100 + InfiniBand"]
print(f"A100+NVLink  c={c_nv:.0f}（论文 1.0K）  s={s_nv / 1e3:.1f}K（论文 6.2K）")
print(f"A100+IB      c={c_ib:.0f}（论文 24.5K） s={s_ib / 1e3:.1f}K（论文 149.5K）")
check(abs(c_nv - 1040) < 10 and abs(s_nv / 1e3 - 6.24) < 0.05, "A100+NVLink 复现论文 Table 2 的 1.0K / 6.2K")
check(abs(c_ib - 24960) < 10 and abs(s_ib / 1e3 - 149.76) < 0.1, "A100+IB 复现论文 Table 2 的 24.5K / 149.5K")

_, _, c_h, s_h = d_["H100 + NVLink"]
_, _, c_hib, s_hib = d_["H100 + IB 400G"]
print(f"H100+NVLink 需要每卡 ≥ {s_h / 1e3:.0f}K token，H100+IB400G 需要 ≥ {s_hib / 1e3:.0f}K token")
check(s_hib / s_h > 15, "同一代 GPU 换到 IB 后门槛抬高一个数量级以上：ring 不该跨机")

A100+NVLink  c=1040（论文 1.0K）  s=6.2K（论文 6.2K）
A100+IB      c=24960（论文 24.5K） s=149.8K（论文 149.5K）
[OK] A100+NVLink 复现论文 Table 2 的 1.0K / 6.2K
[OK] A100+IB 复现论文 Table 2 的 24.5K / 149.5K
H100+NVLink 需要每卡 ≥ 13K token，H100+IB400G 需要 ≥ 237K token
[OK] 同一代 GPU 换到 IB 后门槛抬高一个数量级以上：ring 不该跨机


In [12]:
# MFU 修正：上式用峰值算力；实际注意力有效算力只有峰值的 eta
print(f"{'注意力 MFU':>12} {'A100+NVLink c_min':>20} {'H100+NVLink c_min':>20}")
for eta in [1.0, 0.75, 0.5, 0.25]:
    ca = min_block_len(312e12 * eta, 300e9)
    ch = min_block_len(989e12 * eta, 450e9)
    print(f"{eta:>11.0%} {ca:>20.0f} {ch:>20.0f}")
check(min_block_len(312e12 * 0.5, 300e9) == 520,
      "MFU 减半时最小块长减半（门槛与有效算力成正比）")

     注意力 MFU    A100+NVLink c_min    H100+NVLink c_min
       100%                 1040                 2198
        75%                  780                 1648
        50%                  520                 1099
        25%                  260                  549
[OK] MFU 减半时最小块长减半（门槛与有效算力成正比）


In [13]:
print(f"\n本 notebook 共 {N_ASSERT} 条断言全部通过。")
assert N_ASSERT >= 20, N_ASSERT


本 notebook 共 20 条断言全部通过。
